In [ ]:
import os
import re
import requests
import yaml
import shutil
from pynxtools_apm.examples.get_file_from_archive_formats import get_file_from_zip
from pynxtools_apm.utils.custom_logging import logger

print(os.getcwd())

In [ ]:
def download_requests(url: str) -> str:
    """Query url for dataset, identify the filename, download and return it or empty string."""
    r = requests.get(url, stream=True, allow_redirects=True)
    if r.status_code == 200:
        # logger.debug(f"{r.url}, {r.headers.get('Content-Type')}, {r.headers.get('Content-Disposition')}")
        disposition = r.headers.get("Content-Disposition")
        match = re.search(r'filename="?([^"]+)"?', r.headers.get('Content-Disposition', ""))
        if match:
            output_file_name = f"{match.group(1)}"
        else:  # fall-back
            output_file_name = url.rsplit("/", 1)[-1]
        with open(output_file_name, "wb") as fp:
            for chunk in r.iter_content(1024 * 1024):
                fp.write(chunk)
        return output_file_name
    return ""
# status = download_requests("https://www.repository.cam.ac.uk/bitstreams/341b677d-2d46-483e-85c1-049019c2830d/download")
# status = download_requests("https://zenodo.org/records/17831869/files/2417_Mar-05-2025_12-18_NiC6_Pd109_reconstruction.epos")
# status = download_requests("https://ndownloader.figshare.com/files/21601377")

In [ ]:
with open("data/datasets.yaml", encoding="utf-8") as fp:
    datasets = yaml.safe_load(fp)

for mime_type, examples in datasets.items():
    if not isinstance(examples, dict):
        continue
    for example, metadata in examples.items():
        logger.debug(f"{mime_type}, {example}")
        if not all(concept in metadata for concept in ("name", "spdx")):
            continue
        if "url" in metadata:  # need to download
            if metadata["url"].count(":") == 2:  # possibly compressed
                archive_link, file_path = metadata["url"].rsplit(":", 1)
                logger.debug(
                    f"remote file, compressed >>>> {archive_link}, {file_path}, {file_path.rsplit('/', 1)[-1]} >>>> {metadata['name']}"
                )
                archive_file_name = download_requests(archive_link)
                if archive_file_name.endswith(".zip"):
                    success = get_file_from_zip(archive_file_name, file_path, os.getcwd(), metadata["name"])
                    os.remove(archive_file_name)
            else:
                logger.debug(f"remote file, not compressed >>>> {metadata['url']} >>>> {metadata['name']}")
                data_file_name = download_requests(metadata['url'])
                os.rename(data_file_name, metadata["name"])
        else:
            if os.path.isfile(f"data/{mime_type}/{example}/{metadata['name']}"):
                shutil.copy(f"data/{mime_type}/{example}/{metadata['name']}", f"{metadata['name']}")
                logger.debug(
                    f"local file, not compressed >>>> data/{mime_type}/{example}/{metadata['name']}"
                )